Chargher les données d'oxygène

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Charger les mesures
mesures = pd.read_csv(
    "../data/24_juillet_2025_donnees_preparees.csv"
)

# Sélectionner les quatre capteurs d'oxygène
oxygene = mesures[["Q01", "Q02", "Q03", "Q04"]].copy()

print("Nombre d'instants :", len(oxygene))
display(oxygene.head())

Vérifier les concentrations

In [ ]:
# Convertir les mesures en valeurs numériques
oxygene = oxygene.apply(
    pd.to_numeric,
    errors="coerce"
)

# Afficher les statistiques
display(oxygene.describe())

Créer notre fichier de coordonnées

In [ ]:
import pandas as pd
from pathlib import Path

stations_oxygene = pd.DataFrame({
    "capteur": ["Q01", "Q02", "Q03", "Q04"],
    "nom": [
        "Viangros IN",
        "Senne OUT",
        "Elia",
        "Vétérinaire"
    ],
    "x_lambert72": [
        145351.38972,
        153138.99971,
        150758.96478,
        147265.78987
    ],
    "y_lambert72": [
        167149.68173,
        177763.00027,
        174519.101605,
        169033.91732
    ]
})

fichier = Path("../data/coordonnees_oxygene_senne.csv")

stations_oxygene.to_csv(fichier, index=False)

display(stations_oxygene)
print("Fichier enregistré :", fichier)

Afficher les quatre capteurs sur une première carte

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 9))

ax.scatter(
    stations_oxygene["x_lambert72"],
    stations_oxygene["y_lambert72"],
    s=130,
    color="#247BA0",
    zorder=3
)

for _, station in stations_oxygene.iterrows():
    ax.annotate(
        f'{station["capteur"]} — {station["nom"]}',
        (
            station["x_lambert72"],
            station["y_lambert72"]
        ),
        xytext=(10, 5),
        textcoords="offset points",
        fontsize=10
    )

ax.set_title(
    "Capteurs d'oxygène dissous — Senne",
    fontsize=14
)

ax.set_xlabel("X Lambert 72 (m)")
ax.set_ylabel("Y Lambert 72 (m)")

ax.set_aspect("equal")
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

Préparer les données d'oxygène

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Charger les mesures
mesures = pd.read_csv(
    "../data/24_juillet_2025_donnees_preparees.csv"
)

# Récupérer les valeurs des quatre capteurs
colonnes = stations_oxygene["capteur"].tolist()

oxygene = mesures[colonnes].apply(
    pd.to_numeric,
    errors="coerce"
)

# Convertir en tableau pour l'animation
valeurs_oxygene = oxygene.to_numpy()

print("Dimensions :", valeurs_oxygene.shape)
print("Valeurs minimales et maximales :")
display(oxygene.agg(["min", "max"]))

### Première animation : cercles et concentrations

In [ ]:
# Durée de l'animation
duree = 60
nombre_images = len(valeurs_oxygene)
intervalle = duree * 1000 / nombre_images

# Taille des cercles en fonction de l'oxygène
# La taille est une surface en points² dans matplotlib
def taille_cercles(valeurs):
    valeurs = np.asarray(valeurs, dtype=float)
    return 100 + 65 * np.maximum(valeurs, 0)

# Créer la figure
fig, ax = plt.subplots(figsize=(8, 10))

# Dessiner les cercles à l'instant initial
cercles = ax.scatter(
    stations_oxygene["x_lambert72"],
    stations_oxygene["y_lambert72"],
    s=taille_cercles(valeurs_oxygene[0]),
    color="#3C9EC3",
    alpha=0.35,
    edgecolors="#145A7A",
    linewidths=1.5,
    zorder=3
)

# Ajouter les points centraux des capteurs
ax.scatter(
    stations_oxygene["x_lambert72"],
    stations_oxygene["y_lambert72"],
    s=30,
    color="#145A7A",
    zorder=4
)

# Créer les textes des valeurs
etiquettes = []

for _, station in stations_oxygene.iterrows():
    etiquette = ax.annotate(
        "",
        (
            station["x_lambert72"],
            station["y_lambert72"]
        ),
        xytext=(15, 12),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold"
    )
    etiquettes.append(etiquette)

titre = ax.set_title("", fontsize=14, pad=20)

ax.set_xlabel("X Lambert 72 (m)")
ax.set_ylabel("Y Lambert 72 (m)")
ax.set_aspect("equal")
ax.grid(alpha=0.15)

# Garder la même zone géographique
ax.set_xlim(
    stations_oxygene["x_lambert72"].min() - 2500,
    stations_oxygene["x_lambert72"].max() + 2500
)

ax.set_ylim(
    stations_oxygene["y_lambert72"].min() - 1500,
    stations_oxygene["y_lambert72"].max() + 1500
)

# Fonction de mise à jour
def actualiser_oxygene(image):
    valeurs = valeurs_oxygene[image]

    # Modifier la taille des cercles
    tailles = np.where(
        np.isfinite(valeurs),
        taille_cercles(np.nan_to_num(valeurs)),
        0
    )
    cercles.set_sizes(tailles)

    # Modifier les chiffres
    for i, etiquette in enumerate(etiquettes):
        valeur = valeurs[i]
        capteur = stations_oxygene.iloc[i]["capteur"]

        if np.isfinite(valeur):
            etiquette.set_text(
                f"{capteur} : {valeur:.1f} mg/L"
            )
        else:
            etiquette.set_text(
                f"{capteur} : donnée absente"
            )

    # Afficher la date et l'heure
    date_heure = mesures.iloc[image, 0]

    titre.set_text(
        f"Oxygène dissous dans la Senne\n{date_heure}"
    )

    return (cercles, titre, *etiquettes)

# Créer l'animation
animation_oxygene = FuncAnimation(
    fig,
    actualiser_oxygene,
    frames=nombre_images,
    interval=intervalle,
    blit=False,
    repeat=False
)

plt.close(fig)

print("Animation créée :", nombre_images, "images")

Afficher l'animation

In [ ]:
display(
    HTML(animation_oxygene.to_jshtml())
)

Précisions importantes : les valeurs manquantes feront disparaître les cercles dynamiques, et la taille des cercles est pour l'instant un choix graphique exploratoire, pas un seuil de qualité de l'eau.
Les cercles ne sont pas assez lisibles dans leur changements, ils sont donc que provisoires. Á tester: des barres qui changent de hauteur

Verifier que geo panda est installé

In [ ]:
import geopandas as gpd

print("GeoPandas fonctionne !")
print("Version :", gpd.__version__)

Préparer les coordonnées des capteurs

In [ ]:
from shapely.geometry import Point

capteurs_geo = gpd.GeoDataFrame(
    stations_oxygene.copy(),
    geometry=gpd.points_from_xy(
        stations_oxygene["x_lambert72"],
        stations_oxygene["y_lambert72"]
    ),
    crs="EPSG:31370"
)

display(capteurs_geo[["capteur", "nom", "geometry"]])

### Dessiner le tracé de la senne 

In [ ]:
import geopandas as gpd

url = (
    "https://ows.environnement.brussels/water"
    "?service=WFS"
    "&version=1.1.0"
    "&request=GetFeature"
    "&typeName=water_bodies_surface"
    "&outputFormat=application/json"
)

try:
    cours_eau = gpd.read_file(url)

    print("Données chargées !")
    print("Nombre d'éléments :", len(cours_eau))
    print("Colonnes :", cours_eau.columns.tolist())
    print("Système de coordonnées :", cours_eau.crs)

    display(cours_eau.head())

except Exception as erreur:
    print("Le téléchargement n'a pas fonctionné.")
    print("Erreur :", erreur)

Dessiner la Senne et les quatre capteurs

In [ ]:
# Sélectionner uniquement la Senne
senne = cours_eau[cours_eau["name"] == "Zenne"].copy()

# Convertir dans le même système de coordonnées
# que nos quatre capteurs (Lambert 72)
senne = senne.to_crs("EPSG:31370")

# Créer la figure
fig, ax = plt.subplots(figsize=(9, 11))

# Dessiner le véritable tracé de la Senne
senne.plot(
    ax=ax,
    color="#72B7D2",
    linewidth=4,
    zorder=1
)

# Dessiner les quatre capteurs
capteurs_geo.plot(
    ax=ax,
    color="#145A7A",
    markersize=90,
    edgecolor="white",
    linewidth=1.5,
    zorder=3
)

# Ajouter les noms des capteurs
for _, station in capteurs_geo.iterrows():
    ax.annotate(
        f'{station["capteur"]} — {station["nom"]}',
        (
            station.geometry.x,
            station.geometry.y
        ),
        xytext=(10, 8),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold"
    )

ax.set_title(
    "La Senne et ses quatre capteurs d'oxygène",
    fontsize=15,
    pad=20
)

ax.set_xlabel("X Lambert 72 (m)")
ax.set_ylabel("Y Lambert 72 (m)")
ax.set_aspect("equal")
ax.grid(alpha=0.15)

plt.tight_layout()
plt.show()

Enregistrer le tracé de la Senne

In [ ]:
from pathlib import Path

fichier_senne = Path("../data/tracé_senne.geojson")

senne.to_file(
    fichier_senne,
    driver="GeoJSON"
)

print("Tracé enregistré :", fichier_senne)

### Deuxième animation : barres sur le tracé de la Senne

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Vérifier les données
print("Dimensions :", valeurs_oxygene.shape)

# Déterminer une échelle commune aux quatre capteurs
valeur_max = np.nanmax(valeurs_oxygene)

assert np.isfinite(valeur_max) and valeur_max > 0

# Créer la figure
fig, ax = plt.subplots(figsize=(9, 11))

# Dessiner la Senne
senne.plot(
    ax=ax,
    color="#A8C2CF",
    linewidth=4,
    zorder=1
)

# Dessiner les positions fixes des capteurs
capteurs_geo.plot(
    ax=ax,
    color="#145A7A",
    markersize=45,
    zorder=3
)

# Préparer les barres et les textes
barres = []
etiquettes = []

# Hauteur maximale des barres : 1200 mètres sur la carte
# C'est une hauteur graphique, pas une mesure physique
hauteur_max = 1200

for _, station in capteurs_geo.iterrows():

    x = station.geometry.x
    y = station.geometry.y

    # Barre initialement vide
    barre, = ax.plot(
        [x, x],
        [y, y],
        color="#247BA0",
        linewidth=10,
        solid_capstyle="butt",
        zorder=4
    )

    barres.append(barre)

    # Texte à côté du capteur
    etiquette = ax.annotate(
        "",
        (x, y),
        xytext=(14, 10),
        textcoords="offset points",
        fontsize=10,
        fontweight="bold",
        zorder=5
    )

    etiquettes.append(etiquette)

# Conserver la même échelle pendant l'animation
ax.set_xlim(
    capteurs_geo.total_bounds[0] - 2500,
    capteurs_geo.total_bounds[2] + 2500
)

ax.set_ylim(
    capteurs_geo.total_bounds[1] - 1500,
    capteurs_geo.total_bounds[3] + hauteur_max + 1500
)

ax.set_aspect("equal")
ax.grid(alpha=0.15)

titre = ax.set_title("", fontsize=15, pad=20)

# Fonction qui actualise chaque image
def actualiser_barres(image):

    valeurs = valeurs_oxygene[image]

    for i, station in enumerate(capteurs_geo.itertuples()):

        x = station.geometry.x
        y = station.geometry.y

        valeur = valeurs[i]

        if np.isfinite(valeur):

            hauteur = (
                max(valeur, 0) / valeur_max
            ) * hauteur_max

            # Modifier la hauteur de la barre
            barres[i].set_data(
                [x, x],
                [y, y + hauteur]
            )

            # Modifier le texte
            etiquettes[i].set_text(
                f"{station.capteur} : {valeur:.1f} mg/L"
            )

        else:
            # Ne pas afficher de barre si la donnée manque
            barres[i].set_data([x, x], [y, y])

            etiquettes[i].set_text(
                f"{station.capteur} : donnée absente"
            )

    date_heure = mesures.iloc[image, 0]

    titre.set_text(
        f"Oxygène dissous dans la Senne\n{date_heure}"
    )

    return (*barres, *etiquettes, titre)

# Créer l'animation de 60 secondes
animation_barres = FuncAnimation(
    fig,
    actualiser_barres,
    frames=len(valeurs_oxygene),
    interval=60000 / len(valeurs_oxygene),
    blit=False,
    repeat=False
)

plt.close(fig)

print("Animation créée !")

Afficher l'animation

In [ ]:
display(
    HTML(animation_barres.to_jshtml())
)

Les barres utilisent toutes la même échelle, calculée à partir de la valeur maximale observée. Elles représentent la concentration d'oxygène, et non la hauteur de l'eau.

Exporter la carte animée en vidéo

In [ ]:
%pip install imageio-ffmpeg

Indiquer à Matplotlib où se trouve FFmpeg

In [ ]:
import imageio_ffmpeg
import matplotlib as mpl

chemin_ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()

mpl.rcParams["animation.ffmpeg_path"] = chemin_ffmpeg

print("FFmpeg trouvé :", chemin_ffmpeg)

Exporter l'animation

In [ ]:
from pathlib import Path
from matplotlib.animation import FFMpegWriter

dossier_sortie = Path("../outputs")
dossier_sortie.mkdir(parents=True, exist_ok=True)

fichier_video = dossier_sortie / "senne_oxygene_sans_son.mp4"

writer = FFMpegWriter(
    fps=len(valeurs_oxygene) / 60,
    codec="libx264",
    bitrate=2500
)

animation_barres.save(
    str(fichier_video),
    writer=writer,
    dpi=120
)

print("Vidéo enregistrée :", fichier_video)

Verifier le ficher audio

In [ ]:
from pathlib import Path
from IPython.display import Audio, display

fichier_audio = Path("../audio/senne_oxygene.wav")

print("Le fichier existe :", fichier_audio.exists())

if fichier_audio.exists():
    print("Taille :", fichier_audio.stat().st_size, "octets")
    display(Audio(filename=str(fichier_audio)))

#### Associer la sonification à la vidéo

In [ ]:
from pathlib import Path
import subprocess
import imageio_ffmpeg

# Fichiers existants
video = Path("../outputs/senne_oxygene_sans_son.mp4")
audio = Path("../audio/senne_oxygene.wav")

# Nouvelle vidéo avec son
sortie = Path("../outputs/senne_oxygene_avec_son.mp4")

ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()

subprocess.run([
    ffmpeg,
    "-y",
    "-i", str(video),
    "-i", str(audio),
    "-map", "0:v:0",
    "-map", "1:a:0",
    "-c:v", "copy",
    "-c:a", "aac",
    "-b:a", "192k",
    "-af", "volume=3",
    "-shortest",
    str(sortie)
], check=True)

print("Vidéo avec son créée :", sortie)

Ecouter le resultat

In [ ]:
from IPython.display import Video, display

display(Video(filename="../outputs/senne_oxygene_avec_son.mp4", embed=True))

Charger la video et le son directement, 'est pas possible car torp lourd pour github
- Ici le lein pour la video qui présente l'evolution de la concentration en oxygène dissous dans la Senne le 24 juillet 2025.
- Les quatre stations (Q01, Q02, Q03 et Q04) sont représentées par des barres dont la hauteur varie selon la concentration d'oxygène (mg/L).
- Sonification réalisée avec STRAUSS.
- Le lecteur intégré à VS Code ne restitue pas correctement le son sur notre configuration. Le fichier MP4 a été vérifié avec QuickTime et contient bien une piste audio.
- [▶ Voir la vidéo avec son](https://github.com/IlariaFa/FlowBru-sonification/blob/main/outputs/senne_oxygene_avec_son.mp4)
